In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np

In [3]:
from bayesgpt.simulators import NestedModelFamily
from bayesgpt.simulators.benchmarks import StandardDDM, CollapsingBoundDDM, SuperDDM

In [4]:
# Define global parameter space (superset)
param_names = [
    "v",  # drift
    "a",  # boundary
    "z",  # initial bias
    "tau",  # non-decision time
    "sigma",  # noise scale
    "angle",  # collapse rate
    "s_v",  # All 's' are variability
    "s_a",
    "s_z",
    "s_tau",
    "s_sigma",
    "s_angle",
]

In [8]:
model_family = NestedModelFamily(parameter_names=param_names)

In [9]:
# StandardDDM variants
model_family.add_variant(
    name="std_ddm_1",
    model=StandardDDM,
    free_parameters={
        "v": lambda bs, ctx=None: np.random.normal(0.5, 0.1, bs),
        "a": lambda bs, ctx=None: np.random.uniform(1.0, 2.0, bs),
    },
    fixed_parameters={
        "z": 0.5,
        "tau": 0.3,
        "s_v": 0.0,
        "sigma": 1.0,
        "angle": 0.0,
        "s_z": 0.0,
        "s_tau": 0.0,
    },
)

model_family.add_variant(
    name="std_ddm_2",
    model=StandardDDM,
    free_parameters={
        "v": lambda bs, ctx=None: np.random.normal(1.0, 0.2, bs),
        "s_v": lambda bs, ctx=None: np.random.uniform(0.1, 0.3, bs),
    },
    fixed_parameters={
        "a": 1.5,
        "z": 0.5,
        "tau": 0.4,
        "sigma": 1.0,
        "angle": 0.0,
        "s_z": 0.0,
        "s_tau": 0.0,
    },
)

# CollapsingBoundDDM variants
model_family.add_variant(
    name="coll_ddm_1",
    model=CollapsingBoundDDM,
    free_parameters={
        "v": lambda bs, ctx=None: np.random.normal(0.6, 0.15, bs),
        "angle": lambda bs, ctx=None: np.random.uniform(0.1, 0.5, bs),
    },
    fixed_parameters={
        "a": 1.2,
        "z": 0.5,
        "tau": 0.3,
        "s_v": 0.0,
        "sigma": 1.0,
        "s_z": 0.0,
        "s_tau": 0.0,
    },
)

model_family.add_variant(
    name="coll_ddm_2",
    model=CollapsingBoundDDM,
    free_parameters={
        "a": lambda bs, ctx=None: np.random.uniform(1.5, 2.5, bs),
        "angle": lambda bs, ctx=None: np.random.normal(0.3, 0.05, bs),
    },
    fixed_parameters={
        "v": 0.8,
        "z": 0.5,
        "tau": 0.35,
        "s_v": 0.1,
        "sigma": 1.0,
        "s_z": 0.0,
        "s_tau": 0.0,
    },
)

# SuperDDM variants (scalar v, variabilities)
model_family.add_variant(
    name="super_ddm_1",
    model=SuperDDM,
    free_parameters={
        "v": lambda bs, ctx=None: np.random.normal(0.7, 0.1, bs),
        "s_z": lambda bs, ctx=None: np.random.uniform(0.05, 0.15, bs),
    },
    fixed_parameters={
        "a": 1.8,
        "z": 0.5,
        "tau": 0.3,
        "s_v": 0.2,
        "sigma": 1.0,
        "angle": 0.0,
        "s_tau": 0.0,
    },
)

model_family.add_variant(
    name="super_ddm_2",
    model=SuperDDM,
    free_parameters={
        "angle": lambda bs, ctx=None: np.random.uniform(0.2, 0.4, bs),
        "s_tau": lambda bs, ctx=None: np.random.uniform(0.1, 0.2, bs),
    },
    fixed_parameters={
        "v": 1.0,
        "a": 2.0,
        "z": 0.5,
        "tau": 0.4,
        "s_v": 0.0,
        "sigma": 1.0,
        "s_z": 0.1,
    },
)

In [10]:
batch_size = 100  # Adjust as needed
results = {}
for variant in model_family.variant_names:
    results[variant] = model_family.sample(variant, batch_size)

In [13]:
# Print results
for variant, result in results.items():
    print(f"{variant}:")
    print(f"  RTs: {result['sim_data']['rts'][:5]}")  # First 5 for brevity
    print(f"  Choices: {result['sim_data']['choices'][:5]}")

std_ddm_1:
  RTs: [[1.546]
 [0.636]
 [0.73 ]
 [0.657]
 [1.72 ]]
  Choices: [[0.]
 [1.]
 [1.]
 [1.]
 [1.]]
std_ddm_2:
  RTs: [[0.794]
 [2.015]
 [0.518]
 [0.665]
 [0.674]]
  Choices: [[1.]
 [1.]
 [0.]
 [1.]
 [1.]]
coll_ddm_1:
  RTs: [[1.276]
 [0.972]
 [0.878]
 [2.976]
 [0.688]]
  Choices: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]]
coll_ddm_2:
  RTs: [[1.203]
 [3.344]
 [2.24 ]
 [1.956]
 [1.923]]
  Choices: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]]
super_ddm_1:
  RTs: [[0.79 ]
 [1.635]
 [1.125]
 [1.933]
 [0.734]]
  Choices: [[1.]
 [1.]
 [1.]
 [0.]
 [1.]]
super_ddm_2:
  RTs: [[1.71776559]
 [2.90235961]
 [1.73899522]
 [1.52421245]
 [0.64136946]]
  Choices: [[1.]
 [1.]
 [1.]
 [1.]
 [0.]]


In [11]:
model_family.add_variant(
    name="ddm",
    model=StandardDDM,
    free_parameters={"v": lambda n: np.random.normal(1.0, 0.2, size=n)},
    fixed_parameters={"a": 1.0, "z": 0.5, "tau": 0.3, "s_v": 0.1, "sigma": 1.0},
)

In [12]:
model_family.add_variant(
    name="collapsing",
    model=CollapsingBoundDDM,
    free_parameters={
        "v": lambda n: np.random.normal(1.0, 0.2, size=n),
        "angle": lambda n: np.random.uniform(0.0, 0.5, size=n),
    },
    fixed_parameters={"a": 1.0, "z": 0.5, "tau": 0.3, "s_v": 0.1, "sigma": 1.0},
)

In [17]:
out = model_family.sample("ddm", batch_size=100)
mask = model_family.get_infer_mask("ddm", batch_size=100)
condition = model_family.get_variant_encoder("ddm", batch_size=100)

In [18]:
print(out["sim_data"].shape)
print(out["full_params"].shape)
print(mask.shape)
print(condition.shape)

(100, 2)
(100, 12)
(100, 12)
(100, 3)


In [29]:
out["sim_data"]

array([[0.575, 1.   ],
       [0.428, 1.   ],
       [0.615, 1.   ],
       [0.558, 0.   ],
       [1.13 , 1.   ],
       [0.489, 0.   ],
       [0.401, 0.   ],
       [0.41 , 1.   ],
       [0.974, 1.   ],
       [0.484, 1.   ],
       [0.456, 1.   ],
       [0.633, 0.   ],
       [0.641, 1.   ],
       [0.816, 1.   ],
       [0.599, 1.   ],
       [0.355, 0.   ],
       [0.949, 0.   ],
       [0.625, 1.   ],
       [0.399, 1.   ],
       [0.4  , 1.   ],
       [0.494, 1.   ],
       [0.578, 1.   ],
       [0.558, 1.   ],
       [0.842, 1.   ],
       [0.532, 1.   ],
       [0.425, 1.   ],
       [0.382, 1.   ],
       [0.49 , 1.   ],
       [1.3  , 1.   ],
       [0.722, 0.   ],
       [0.751, 1.   ],
       [0.708, 1.   ],
       [0.601, 1.   ],
       [0.775, 0.   ],
       [0.445, 1.   ],
       [1.065, 1.   ],
       [0.586, 1.   ],
       [0.383, 1.   ],
       [0.745, 1.   ],
       [0.378, 1.   ],
       [0.375, 1.   ],
       [0.449, 0.   ],
       [1.404, 1.   ],
       [0.4

In [30]:
out["full_params"]

array([[0.554168  , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.96912664, 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.80194896, 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [1.355698  , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.5866667 , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [1.0332814 , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.95180297, 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.9996534 , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [1.2273102 , 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0.        ],
       [0.83825034, 1.        , 0.5       , 0.3       , 0.1       ,
        1.        , 0. 